# Chapter `1.5` - Personal Chef 🧑‍🍳

## Setup

### Module imports

In [ ]:
# Basic utils.
from os import getenv
from dotenv import load_dotenv

# i/o
from ipywidgets import FileUpload
from typing import Dict, Any
import base64

# o/p formatting
from IPython.display import display, Markdown
from pprint import pprint

# LC modules
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool

from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

from tavily import TavilyClient

### **Gemini** API setup

In [2]:
load_dotenv()

GOOGLE_API_KEY = getenv("GOOGLE_API_KEY")
GEMINI_API_MODEL = getenv("GEMINI_API_MODEL")

model = ChatGoogleGenerativeAI(model=GEMINI_API_MODEL, api_key=GOOGLE_API_KEY)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Web Search Tool

### Setting up the **Tavily** API

In [3]:
tavily_client = TavilyClient()

@tool("web_search", description="Search the web for the most relevant information that can fulfill the user's request.")
def web_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query)

## Agent Setup

In [4]:
system_prompt = """
You are a personal chef agent. The user will give you a list of ingredients they have left over in their house.

Using the web search tool provided to you, search the web for recipes that can be made with the ingredients that they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

### Testing multiple modalities

### Uploading an image
![Inside of my refrigerator](../../pics/inside_of_fridge.png)

In [5]:
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
pprint(uploader.value)

({'content': <memory at 0x000002337DDE7A00>,
  'last_modified': datetime.datetime(2026, 1, 9, 16, 10, 56, 338000, tzinfo=datetime.timezone.utc),
  'name': 'inside_of_fridge.png',
  'size': 28476,
  'type': 'image/png'},)


### Encoding the image to `base64`

In [7]:
# Fetch the FIRST uploaded file
uploaded_file = uploader.value[0]

# This is a memoryview object
content_mv = uploaded_file["content"]

# Convert memoryview object -> bytes object
img_bytes = content_mv.tobytes()

# Now, perform base64 encoding
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

### Prompting

In [9]:
multimodal_question = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "This is what I have left in my refrigerator. What can I make? Give me the detailed recipe instructions.",
        },
        {"type": "image", "base64": img_b64, "mime_type": "image/png"},
    ]
)
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [multimodal_question]}, config)
Markdown(response["messages"][-1].content)

Here are the recipe instructions for Nana's Butter Cookies with Milk-Jam Filling:

**Yields:** about 3 dozen cookies
**Prep time:** 30 minutes
**Bake time:** 10-12 minutes

**Ingredients:**

*   1 1/2 cups (3 sticks) unsalted butter, room temperature
*   2/3 cup powdered sugar
*   2 teaspoons vanilla extract
*   2 tablespoons whole milk
*   1 large egg
*   1 quart goat's milk or cow's milk (for filling)
*   Fruit preserves or jam of your choice (for filling)

**Instructions:**

**For the Cookies:**

1.  **Cream butter and sugar:** In a large bowl, use an electric mixer to beat the softened butter and powdered sugar together until light and fluffy, about 4 minutes.
2.  **Add wet ingredients:** Beat in the vanilla extract, 2 tablespoons of whole milk, and the egg until well combined.
3.  **Add flour:** Gradually add the flour, mixing on low speed until just combined. Be careful not to overmix.
4.  **Chill the dough:** Shape the dough into a disc, wrap it in plastic wrap, and refrigerate for at least 1 hour, or until firm enough to roll.
5.  **Preheat oven:** Preheat your oven to 350°F (175°C). Line baking sheets with parchment paper.
6.  **Shape the cookies:** On a lightly floured surface, roll out the dough to about 1/4-inch thickness. Use cookie cutters to cut out shapes. Place the cookies on the prepared baking sheets.
7.  **Bake:** Bake for 10-12 minutes, or until the edges are lightly golden. Let the cookies cool on the baking sheets for a few minutes before transferring them to a wire rack to cool completely.

**For the Milk-Jam Filling:**

1.  **Warm the milk:** In a small saucepan, gently warm the quart of milk over low heat. Do not boil.
2.  **Prepare the jam:** Have your fruit preserves or jam ready.

**Assembly:**

1.  **Fill the cookies:** Once the cookies are completely cool, spread a layer of jam on the bottom of one cookie.
2.  **Dip in milk:** Dip the jam-covered side of the cookie into the warmed milk.
3.  **Top with another cookie:** Place another cookie on top, jam-side down, to create a sandwich.
4.  **Repeat:** Continue with the remaining cookies.

Enjoy your homemade Nana's Butter Cookies with Milk-Jam Filling!